In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [35]:
url = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
df = pd.read_csv(url)
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [36]:
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]
df.head()


,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,170,159.0,3413.433759,2003,13.231729
1,130,97.0,3149.664934,2007,13.688217
2,170,78.0,3079.038997,2018,14.246341
3,220,NaN,2542.392402,2009,16.912736
4,210,140.0,3460.870990,2009,12.488369


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9704 entries, 0 to 9703
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  9704 non-null   int64  
 1   horsepower           8996 non-null   float64
 2   vehicle_weight       9704 non-null   float64
 3   model_year           9704 non-null   int64  
 4   fuel_efficiency_mpg  9704 non-null   float64
dtypes: float64(3), int64(2)
memory usage: 379.2 KB


Question 1. Missing values

In [38]:
df.isnull().sum()

engine_displacement      0
horsepower             708
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Question 2. Median for horse power 

In [39]:
df['horsepower'].median()


np.float64(149.0)

Shuffle and Split Dataset

In [40]:
from sklearn.model_selection import train_test_split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n=len(df)

n_train = int(0.6*n)
n_val = int(0.2*n)
n_test = n - n_train - n_val

df_train = df.iloc[:n_train]
df_val = df.iloc[n_train:n_train+n_val]
df_test = df.iloc[n_train+n_val:]

Prepare features

In [41]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

X_train = df_train.drop(columns=['fuel_efficiency_mpg'])
X_val = df_val.drop(columns=['fuel_efficiency_mpg'])
X_test = df_test.drop(columns=['fuel_efficiency_mpg'])

Fill with 0

In [42]:
X_train_0 = X_train.fillna(0)
X_val_0 = X_val.fillna(0)

Fill with Mean

In [43]:
mean_horsepower = X_train['horsepower'].mean()

X_train_mean = X_train.fillna({'horsepower': mean_horsepower})
X_val_mean = X_val.fillna({'horsepower': mean_horsepower})

RMSE (fill 0)

In [45]:
lr_0 = LinearRegression()
lr_0.fit(X_train_0, y_train)

y_pred_0 = lr_0.predict(X_val_0)
rmse_0 = np.sqrt(mean_squared_error(y_val, y_pred_0))
print("RMSE (fill 0):", round(rmse_0, 2))

RMSE (fill 0): 0.52


RMSE (fill mean)

In [46]:
lr_mean = LinearRegression()
lr_mean.fit(X_train_mean, y_train)

y_pred_mean = lr_mean.predict(X_val_mean)
rmse_mean = np.sqrt(mean_squared_error(y_val, y_pred_mean))
print("RMSE (fill mean):", round(rmse_mean, 2))


RMSE (fill mean): 0.46


In [48]:
X_train_reg = X_train.fillna(0)
X_val_reg = X_val.fillna(0)

Question 4. Best regularization

In [50]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
rmse_scores = {}

for r in r_values:
    model = Ridge(alpha=r)  
    model.fit(X_train_reg, y_train)  
    y_pred = model.predict(X_val_reg) 
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))  
    rmse_scores[r] = round(rmse, 2)
    print(f"r = {r}: RMSE = {round(rmse, 2)}")

r = 0: RMSE = 0.52
r = 0.01: RMSE = 0.52
r = 0.1: RMSE = 0.52
r = 1: RMSE = 0.52
r = 5: RMSE = 0.52
r = 10: RMSE = 0.52
r = 100: RMSE = 0.52


In [51]:
X = df.drop(columns=['fuel_efficiency_mpg'])
y = df['fuel_efficiency_mpg']

Question 5. RMSE Standard Deviation

In [52]:
seeds = [0,1,2,3,4,5,6,7,8,9]
rmse_list = []

for seed in seeds:
    df_shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    n = len(df_shuffled)
    n_train = int(0.6 * n)
    n_val = int(0.2 * n)
    
    df_train = df_shuffled.iloc[:n_train]
    df_val = df_shuffled.iloc[n_train:n_train+n_val]
    
    X_train = df_train.drop(columns=['fuel_efficiency_mpg']).fillna(0)
    y_train = df_train['fuel_efficiency_mpg'].values
    
    X_val = df_val.drop(columns=['fuel_efficiency_mpg']).fillna(0)
    y_val = df_val['fuel_efficiency_mpg'].values
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    rmse_list.append(rmse)
    
rmse_array = np.array(rmse_list)

std_rmse = np.std(rmse_array)
print("RMSE for each seed:", np.round(rmse_array, 3))
print("Standard deviation of RMSE:", round(std_rmse, 3))

RMSE for each seed: [0.516 0.509 0.516 0.527 0.533 0.518 0.513 0.53  0.507 0.521]
Standard deviation of RMSE: 0.008


In [53]:
df_shuffled = df.sample(frac=1, random_state=9).reset_index(drop=True)

n = len(df_shuffled)
n_train = int(0.6 * n)
n_val = int(0.2 * n)
n_test = n - n_train - n_val

df_train = df_shuffled.iloc[:n_train]
df_val = df_shuffled.iloc[n_train:n_train+n_val]
df_test = df_shuffled.iloc[n_train+n_val:]

In [54]:
df_train_combined = pd.concat([df_train, df_val]).reset_index(drop=True)

In [55]:
X_train = df_train_combined.drop(columns=['fuel_efficiency_mpg']).fillna(0)
y_train = df_train_combined['fuel_efficiency_mpg'].values

X_test = df_test.drop(columns=['fuel_efficiency_mpg']).fillna(0)
y_test = df_test['fuel_efficiency_mpg'].values

Question 6. Evaluation on test

In [56]:
model = Ridge(alpha=0.001)  # r = 0.001
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", round(rmse_test, 3))

Test RMSE: 0.529
